# GrowWithMe — Offline Maternal Risk Model (Tier 1)

Trains a tiny missing-input-tolerant risk classifier on the Maternal Health Risk dataset
and exports a quantized **TFLite** model (~50–200 KB) + `manifest.json` for the
GrowWithMe model registry.

**Run top to bottom on Google Colab (CPU is fine, ~5 minutes).**

- Inputs: age, systolic BP, diastolic BP, blood sugar (mmol/L), body temp (°F), heart rate
- Any input may be MISSING — the model trains with random masking + missingness flags,
  so a caregiver with only age + temp + ANC-card BP still gets a (lower-confidence) estimate.
- Output: P(low), P(mid), P(high)
- The metric that matters most: **recall on the high-risk class**.

In [ ]:
%pip -q install tensorflow scikit-learn pandas numpy requests
import hashlib, json, os
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
np.random.seed(42); tf.random.set_seed(42)
print('TF', tf.__version__)

## 1. Load + clean (deduplicate BEFORE splitting — the UCI set has many duplicate rows)

In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00639/Maternal%20Health%20Risk%20Data%20Set.csv'
df = pd.read_csv(URL)
df.columns = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar', 'body_temp', 'heart_rate', 'risk']
print('raw rows:', len(df))
df = df.drop_duplicates().reset_index(drop=True)
print('after dedupe:', len(df))
# One known bad row in this dataset: heart rate of 7 — physiologically impossible.
df = df[df.heart_rate > 30].reset_index(drop=True)
CLASSES = ['low risk', 'mid risk', 'high risk']
df['label'] = df.risk.map({c: i for i, c in enumerate(CLASSES)})
FEATURES = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar', 'body_temp', 'heart_rate']
print(df.risk.value_counts())
df.describe()

In [ ]:
X = df[FEATURES].values.astype('float32')
y = df.label.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42)
MEAN = X_tr.mean(axis=0); STD = X_tr.std(axis=0)
print('train/val/test:', len(X_tr), len(X_va), len(X_te))
print('mean:', MEAN.round(2), '\nstd: ', STD.round(2))

## 2. Missing-input tolerance
Model input = 12 values: 6 normalized features (0 where missing) + 6 flags (1 = value present).
Training data is augmented with random masks, weighted toward the caregiver-realistic pattern
(blood sugar / heart rate unknown, BP from the ANC card).

In [ ]:
def encode(X, mask):
    """X raw values, mask 1=present. Returns [normalized*mask, mask]."""
    Xn = (X - MEAN) / STD
    return np.concatenate([Xn * mask, mask], axis=1).astype('float32')

def augment(X, y, copies=6):
    """Full-input copy + masked copies. Age is always known."""
    outX, outY = [encode(X, np.ones_like(X))], [y]
    caregiver = np.array([1, 1, 1, 0, 1, 0], dtype='float32')  # age, BP, temp — no BS/HR
    for _ in range(copies):
        mask = (np.random.rand(*X.shape) > 0.35).astype('float32')
        mask[:, 0] = 1  # age always present
        # Half of the masked copies follow the caregiver pattern exactly
        rows = np.random.rand(len(X)) < 0.5
        mask[rows] = caregiver
        outX.append(encode(X, mask)); outY.append(y)
    return np.concatenate(outX), np.concatenate(outY)

Xa_tr, ya_tr = augment(X_tr, y_tr)
Xe_va = encode(X_va, np.ones_like(X_va))
print('augmented train:', Xa_tr.shape)

## 3. Train (class-weighted toward high-risk recall)

In [ ]:
w = compute_class_weight('balanced', classes=np.arange(3), y=y_tr)
w[2] *= 1.5  # extra weight: missing a high-risk mother is the costly error
class_weight = dict(enumerate(w))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(12,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.15),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(Xa_tr, ya_tr, validation_data=(Xe_va, y_va), epochs=120, batch_size=64,
          class_weight=class_weight, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)])
print('val accuracy:', model.evaluate(Xe_va, y_va, verbose=0)[1].round(3))

## 4. Evaluate BOTH conditions: responder (all 6) and caregiver (partial)

In [ ]:
def report(name, mask_row):
    mask = np.tile(mask_row, (len(X_te), 1)).astype('float32')
    pred = model.predict(encode(X_te, mask), verbose=0).argmax(axis=1)
    print(f'\n=== {name} ===')
    print(classification_report(y_te, pred, target_names=CLASSES, digits=3))
    print(confusion_matrix(y_te, pred))

report('RESPONDER — all six vitals', np.ones(6))
report('CAREGIVER — age, BP, temp only', np.array([1, 1, 1, 0, 1, 0]))
report('MINIMAL — age + BP only', np.array([1, 1, 1, 0, 0, 0]))

**Check before shipping:** high-risk recall should be ≥0.85 with all six vitals and degrade
gracefully (not collapse) with partial inputs. If the caregiver condition collapses,
increase `copies` in `augment` or the caregiver-pattern share and retrain.

## 5. Export TFLite (quantized) + manifest.json

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quantization
tfl = conv.convert()
open('maternal_risk.tflite', 'wb').write(tfl)
sha = hashlib.sha256(tfl).hexdigest()

manifest = {
    'name': 'maternal-risk',
    'version': 1,
    'kind': 'tflite',
    'sizeBytes': len(tfl),
    'sha256': sha,
    'classes': CLASSES,
    'features': FEATURES,
    'featureLabels': ['Age (years)', 'Systolic BP (mmHg)', 'Diastolic BP (mmHg)',
                      'Blood sugar (mmol/L)', 'Body temperature (°F)', 'Heart rate (bpm)'],
    'validRanges': {'age': [10, 70], 'systolic_bp': [60, 220], 'diastolic_bp': [40, 140],
                    'blood_sugar': [3, 25], 'body_temp': [95, 106], 'heart_rate': [40, 200]},
    'mean': MEAN.tolist(),
    'std': STD.tolist(),
    'inputLayout': 'six normalized values (0 where missing) followed by six presence flags',
    'trainedOn': 'UCI Maternal Health Risk (Bangladesh, deduplicated), local rows: 0',
    'disclaimer': 'Decision support only — may be wrong. Never downgrades the danger-sign rules.',
}
json.dump(manifest, open('manifest.json', 'w'), indent=2)
print(f'maternal_risk.tflite: {len(tfl)/1024:.0f} KB\nsha256: {sha}')
print(json.dumps(manifest, indent=2)[:400], '...')

## 6. Sanity-check the TFLite file (must match Keras predictions)

In [ ]:
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
inp, out = it.get_input_details()[0], it.get_output_details()[0]
def tflite_predict(x):
    it.set_tensor(inp['index'], x.reshape(1, -1)); it.invoke()
    return it.get_tensor(out['index'])[0]

sample = encode(X_te[:5], np.ones((5, 6), dtype='float32'))
for i in range(5):
    p = tflite_predict(sample[i])
    print(f'true={CLASSES[y_te[i]]:9s}  tflite={CLASSES[p.argmax()]:9s}  probs={p.round(3)}')

## 7. Upload to Cloudinary + register with the backend
Fill in your credentials (same ones as the backend `.env`), run, then paste the printed
`curl` command in a terminal (or run the last cell) to register the model so both apps
can discover it at `GET /api/v1/models/maternal-risk`.

In [ ]:
import requests
CLOUD_NAME = ''   # <-- your Cloudinary cloud name
UPLOAD_PRESET = ''  # <-- an UNSIGNED upload preset (create in Cloudinary settings)

up = requests.post(
    f'https://api.cloudinary.com/v1_1/{CLOUD_NAME}/raw/upload',
    files={'file': ('maternal_risk_v1.tflite', tfl)},
    data={'upload_preset': UPLOAD_PRESET, 'public_id': 'growwithme/models/maternal_risk_v1'},
).json()
print(up.get('secure_url') or up)
manifest['url'] = up['secure_url']
json.dump(manifest, open('manifest.json', 'w'), indent=2)

In [ ]:
# Register with the backend (needs an ADMIN account's access token).
BACKEND = 'https://growwithme.onrender.com/api/v1'
ADMIN_TOKEN = ''  # <-- paste an admin accessToken
r = requests.post(f'{BACKEND}/admin/models', json=manifest,
                  headers={'Authorization': f'Bearer {ADMIN_TOKEN}'})
print(r.status_code, r.json())